# SafeBank Assist — Colab run (Group 22)

PCAM ZG516 Applied Generative AI · BANKING77 · `Qwen/Qwen2.5-1.5B-Instruct` (baseline: `gpt2`)

Runs top to bottom on a **free T4** (Runtime → Change runtime type → T4 GPU). This notebook covers environment and reproducibility, the dataset and its preprocessing record, the **E1 baseline**, the tokenizer walkthrough, and an **E7** smoke run. The E2/E3/E6 sweeps build on the objects created here.

Everything written to `artifacts/results/` and `artifacts/figures/` is report evidence.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No GPU: switch the runtime to T4" 

In [ ]:
# Clone the repo (if it is private, use https://<token>@github.com/... or upload a zip).
# Re-running this cell updates an existing clone to the tip of BRANCH, so a stale
# runtime never runs old code. Restart the runtime afterwards if safebank was already imported.
# Outside Colab the notebook runs against the checkout it lives in.
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/2025paml539-lab/safebank-assist-group22.git"
BRANCH = "main"
if IN_COLAB:
    REPO_DIR = "/content/safebank-assist-group22"
    if not os.path.exists(REPO_DIR):
        !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    REPO_DIR = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
%cd {REPO_DIR}
!git log -1 --format='%h %s'

In [ ]:
# Only the packages Colab does not already pin. torch is Colab's preinstalled build;
# its exact version is recorded by device_report() below. Locally, install requirements.txt instead.
if IN_COLAB:
    %pip install -q -r requirements-colab.txt

In [ ]:
import sys, json
sys.path.insert(0, "src")

from safebank import config
from safebank.model_loader import device_report

for d in (config.RESULTS_DIR, config.FIGURES_DIR, config.DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

report = device_report().as_dict()
report.update(seed=config.SEED, model_id=config.MODEL_ID, baseline_model_id=config.BASELINE_MODEL_ID)
(config.RESULTS_DIR / "device_report.json").write_text(json.dumps(report, indent=2))
report

## 2. Dataset: BANKING77

Loads the official PolyAI CSVs with the Hub's canonical label order (the Hub dataset is script-based, which `datasets` ≥ 4 refuses). `load_banking77` runs the mandatory `validate_label_space()` check.

In [ ]:
from safebank.data_prep import (
    load_banking77, dataset_label_space, class_distribution,
    build_eval_sample, build_example_bank, preprocessing_record,
)

train, test = load_banking77()
LABELS = dataset_label_space(train)
print(train, test, sep="\n")
print(len(LABELS), "intents, e.g.", LABELS[:5])

In [ ]:
eval_rows = build_eval_sample(test, LABELS)            # 5 x 77 = 385, seed 42, TEST split
EXAMPLES = build_example_bank(                         # TRAIN split, never an eval query
    train, LABELS, exclude={r["query"] for r in eval_rows}
)
record = preprocessing_record(train, test, eval_rows)

with open(config.RESULTS_DIR / "eval_sample.jsonl", "w") as f:
    for r in eval_rows:
        f.write(json.dumps(r) + "\n")
(config.RESULTS_DIR / "preprocessing_record.json").write_text(json.dumps(record, indent=2))
(config.RESULTS_DIR / "few_shot_examples.json").write_text(json.dumps(EXAMPLES, indent=2))
record

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dist = pd.DataFrame({
    "train": class_distribution(train, LABELS),
    "test": class_distribution(test, LABELS),
}).sort_values("train", ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(dist)), dist["train"], color="#4C72B0")
ax.set_xticks(range(len(dist)), dist.index, rotation=90, fontsize=6)
ax.set_ylabel("train rows")
ax.set_title(f"BANKING77 train class distribution (min {dist.train.min()}, max {dist.train.max()}; test is 40 per intent)")
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "class_distribution.png", dpi=200)
dist.describe()

In [ ]:
# Representative examples table for report section 3
pd.DataFrame(eval_rows).groupby("gold_intent").head(1).head(10)

## 3. E1 — baseline: one simple prompt, GPT-2 vs Qwen

One deliberately plain prompt, no schema, no examples, greedy decoding, the same 5 queries for both models. GPT-2 has no chat template and was not instruction-tuned; that asymmetry is the point of the baseline.

In [ ]:
from safebank.generator import DecodeConfig
from safebank.model_loader import load_model_and_tokenizer, HFGenerator, clear_cache

E1_PROMPT = "Customer message: {query}\nWhat does the customer want? Answer in one short sentence.\nAnswer:"
e1_rows = eval_rows[::77][:5]          # 5 different intents, fixed by the seed
greedy = DecodeConfig(max_new_tokens=48)

def run_e1(model_id):
    gen = HFGenerator(*load_model_and_tokenizer(model_id))
    outs = [gen.generate(E1_PROMPT.format(query=r["query"]), greedy) for r in e1_rows]
    return outs, gen.usage_log

gpt2_out, gpt2_usage = run_e1(config.BASELINE_MODEL_ID)
clear_cache()
qwen_out, qwen_usage = run_e1(config.MODEL_ID)

e1 = pd.DataFrame({
    "query": [r["query"] for r in e1_rows],
    "gold_intent": [r["gold_intent"] for r in e1_rows],
    "gpt2": gpt2_out,
    "qwen2.5-1.5b-instruct": qwen_out,
    "gpt2_ms": [u.latency_ms for u in gpt2_usage],
    "qwen_ms": [u.latency_ms for u in qwen_usage],
})
e1.to_csv(config.RESULTS_DIR / "e1_baseline.csv", index=False)
pd.set_option("display.max_colwidth", 120)
e1

## 4. Tokenization walkthrough (report §4b)

Real tokenizer output for one query: the text → token ids → the model's vocabulary pieces. Qwen stays loaded from E1.

In [ ]:
model, tok = load_model_and_tokenizer(config.MODEL_ID)
q = e1_rows[0]["query"]
ids = tok(q)["input_ids"]
print(q)
print(len(ids), "tokens:", ids)
print([tok.decode([i]) for i in ids])
print("vocab size:", tok.vocab_size, "| hidden size:", model.config.hidden_size,
      "| layers:", model.config.num_hidden_layers, "| heads:", model.config.num_attention_heads,
      "| context:", model.config.max_position_embeddings)

## 5. E7 — integrated pipeline smoke run

The real `HFGenerator` replaces `FakeGenerator`; nothing else in the pipeline changes.

In [ ]:
from safebank.pipeline import SafeBankPipeline
from safebank.prompts import PromptStrategy

qwen = HFGenerator(model, tok)
pipe = SafeBankPipeline(qwen, LABELS, strategy=PromptStrategy.ZERO_SHOT)

for query in [
    "My new card still hasn't arrived, it's been two weeks",
    "I lost my card on the train this morning",
    "Is there an ATM near 560001?",
]:
    r = pipe.run(query)
    print(query, "\n  ->", json.dumps(r.to_row(), default=str)[:400], "\n  ", r.final_text, "\n")

## 6. Handover for E2 / E3 / E6

Objects ready for the sweeps: `LABELS`, `eval_rows` (385), `EXAMPLES` (train-split few-shot bank), `qwen` (`HFGenerator`, also has `generate_batch` for speed) and `qwen.usage_log` for token counts (criterion 5).

For the few-shot variant, pass the train-split bank through the pipeline: `SafeBankPipeline(qwen, LABELS, strategy=PromptStrategy.FEW_SHOT, prompt_kwargs={"examples": EXAMPLES})`. Without it the pipeline warns and falls back to the illustrative offline bank, which must not be used for reported E2 numbers.